# Master Pipeline — End-to-End Medallion Orchestration

In [ ]:
# ============================================================
#  Master Pipeline Orchestration  —  End-to-End Medallion Run
# ============================================================
#  Order:
#    Bronze   →  Silver   →  Gold Dims  →  Gold Facts
# ============================================================
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")

base = ".."  # adjust if run from outside Notebooks/orchestration

# ---------- Bronze ----------
bronze_nbs = [
    "bronze/nb_customer_files_bronze",
    "bronze/nb_sales_files_bronze",
    "bronze/nb_discount_files_bronze",
    "bronze/nb_region_files_bronze",
    "bronze/nb_product_files_bronze",
    "bronze/nb_store_files_bronze",
    "bronze/nb_invoice_srcfiles_aws_bronze",
]
for nb in bronze_nbs:
    print(f"  Running {nb} ...")
    dbutils.notebook.run(f"{base}/{nb}", 600, {"environment": env})

# ---------- Silver ----------
silver_nbs = [
    "silver/nb_customer_bronze_silver",
    "silver/nb_sales_bronze_silver",
    "silver/nb_discount_bronze_silver",
    "silver/nb_region_bronze_silver",
    "silver/nb_product_bronze_silver",
    "silver/nb_store_bronze_silver",
    "silver/nb_invoice_aws_bronze_silver",
]
for nb in silver_nbs:
    print(f"  Running {nb} ...")
    dbutils.notebook.run(f"{base}/{nb}", 600, {"environment": env})

# ---------- Gold Dimensions ----------
dim_nbs = [
    "gold/nb_dim_date_load",
    "gold/nb_dim_region_scd1_pyspark",
    "gold/nb_dim_product_scd2_pyspark",
    "gold/nb_dim_store_scd2_pyspark",
    "gold/nb_dim_discount_scd2_pyspark",
    "gold/nb_cust_silver_gold",          # existing customer SCD2
]
for nb in dim_nbs:
    print(f"  Running {nb} ...")
    dbutils.notebook.run(f"{base}/{nb}", 600, {"environment": env})

# ---------- Gold Facts ----------
fact_nbs = [
    "gold/nb_fact_sales_load",
    "gold/nb_fact_invoice_load",
]
for nb in fact_nbs:
    print(f"  Running {nb} ...")
    dbutils.notebook.run(f"{base}/{nb}", 600, {"environment": env})

print("End-to-end Medallion pipeline complete.")
